# Cell Type Annotation

This notebook annotates the clustered `s4d8` single-cell RNA-seq dataset using three complementary strategies:

1. marker-based manual annotation,
2. automated annotation with CellTypist,
3. reference mapping and weighted KNN label transfer with scArches.

The workflow follows the *Single-cell Best Practices* annotation chapter while using local files instead of LaminDB.

### Expected repository structure

```text
data/
└── s4d8_clustered.h5ad

reference/
├── annotation_reference_features.csv
├── annotation_reference_model.pt
└── annotation_reference_embedding.h5ad

results/
└── s4d8_annotated.h5ad
```

Large data/model files can be kept outside Git if needed; the notebook uses repository-relative paths for reproducibility.


In [ ]:
import shutil
import sys
from pathlib import Path

import celltypist
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas.core.indexes.base as pandas_indexes_base
import scanpy as sc
import scarches as sca
import seaborn as sns
from celltypist import models
from scipy.sparse import csr_matrix

# Repository-relative paths
DATA_DIR = Path("data")
REFERENCE_DIR = Path("reference")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Setup

In [ ]:
sc.set_figure_params(figsize=(5, 5))

## 2. Load the clustered dataset

In [ ]:
adata = sc.read_h5ad(
    DATA_DIR / "s4d8_clustered.h5ad"
)

adata


## 3. Marker-based manual annotation

Define marker genes and inspect known cell populations in the query dataset.

In [ ]:
# keys: cell types or populations, values: lists of marker genes
marker_genes = {
    "CD14+ Mono": ["FCN1", "CD14"],
    "CD16+ Mono": ["TCF7L2", "FCGR3A", "LYN"],
    "ID2-hi myeloid prog": [
        "CD14",
        "ID2",
        "VCAN",
        "S100A9",
        "CLEC12A",
        "KLF4",
        "PLAUR",
    ],
    "cDC1": ["CLEC9A", "CADM1"],
    "cDC2": [
        "CST3",
        "COTL1",
        "LYZ",
        "DMXL2",
        "CLEC10A",
        "FCER1A",
    ],  # Note: DMXL2 should be negative
    "Normoblast": ["SLC4A1", "SLC25A37", "HBB", "HBA2", "HBA1", "TFRC"],
    "Erythroblast": ["MKI67", "HBA1", "HBB"],
    "Proerythroblast": [
        "CDK6",
        "SYNGR1",
        "HBM",
        "GYPA",
    ],  # Note HBM and GYPA are negative markers
    "NK": ["GNLY", "NKG7", "CD247", "GRIK4", "FCER1G", "TYROBP", "KLRG1", "FCGR3A"],
    "ILC": ["ID2", "PLCG2", "GNLY", "SYNE1"],
    "Lymph prog": [
        "VPREB1",
        "MME",
        "EBF1",
        "SSBP2",
        "BACH2",
        "CD79B",
        "IGHM",
        "PAX5",
        "PRKCE",
        "DNTT",
        "IGLL1",
    ],
    "Naive CD20+ B": ["MS4A1", "IL4R", "IGHD", "FCRL1", "IGHM"],
    "B1 B": [
        "MS4A1",
        "SSPN",
        "ITGB1",
        "EPHA4",
        "COL4A4",
        "PRDM1",
        "IRF4",
        "CD38",
        "XBP1",
        "PAX5",
        "BCL11A",
        "BLK",
        "IGHD",
        "IGHM",
        "ZNF215",
    ],  # Note IGHD and IGHM are negative markers
    "Transitional B": ["MME", "CD38", "CD24", "ACSM3", "MSI2"],
    "Plasma cells": ["MZB1", "HSP90B1", "FNDC3B", "PRDM1", "IGKC", "JCHAIN"],
    "Plasmablast": ["XBP1", "RF4", "PRDM1", "PAX5"],  # Note PAX5 is a negative marker
    "CD4+ T activated": ["CD4", "IL7R", "TRBC2", "ITGB1"],
    "CD4+ T naive": ["CD4", "IL7R", "TRBC2", "CCR7"],
    "CD8+ T": ["CD8A", "CD8B", "GZMK", "GZMA", "CCL5", "GZMB", "GZMH", "GZMA"],
    "T activation": ["CD69", "CD38"],  # CD69 much better marker!
    "T naive": ["LEF1", "CCR7", "TCF7"],
    "pDC": ["GZMB", "IL3RA", "COBLL1", "TCF4"],
    "G/M prog": ["MPO", "BCL2", "KCNQ5", "CSF3R"],
    "HSC": ["NRIP1", "MECOM", "PROM1", "NKAIN2", "CD34"],
    "MK/E prog": [
        "ZNF385D",
        "ITGA2B",
        "RYR3",
        "PLCB1",
    ],  # Note PLCB1 is a negative marker
}

In [ ]:
marker_genes_in_data = {}
for ct, markers in marker_genes.items():
    markers_found = []
    for marker in markers:
        if marker in adata.var.index:
            markers_found.append(marker)
    marker_genes_in_data[ct] = markers_found

In [ ]:
adata.layers["counts"] = adata.layers["soupX_counts"].copy()
adata.X = adata.layers["scran_normalization"].copy()

In [ ]:
adata.var["highly_variable"] = adata.var["highly_deviant"]

In [ ]:
sc.tl.pca(
    adata,
    n_comps=50,
    use_highly_variable=True
)

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.umap(adata)

### 3.1 B-cell and plasma-cell marker inspection

In [ ]:
B_plasma_cts = [
    "Naive CD20+ B",
    "B1 B",
    "Transitional B",
    "Plasma cells",
    "Plasmablast",
]

In [ ]:
for ct in B_plasma_cts:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adata,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax="p99",
        sort_order=False,
        frameon=False,
        cmap="Reds",
    )
    print("\n\n\n")

### 3.2 Leiden clustering at two resolutions

In [ ]:
sc.tl.leiden(
    adata,
    resolution=1,
    key_added="leiden_1"
)

In [ ]:
sc.pl.umap(
    adata,
    color="leiden_1"
)

In [ ]:
sc.tl.leiden(
    adata,
    resolution=2,
    key_added="leiden_2"
)

In [ ]:
sc.pl.umap(
    adata,
    color="leiden_2",
    legend_loc="on data"
)

In [ ]:
sc.pl.umap(
    adata,
    color="leiden_1",
    legend_loc="on data"
)

### 3.3 Manual B-cell annotation

In [ ]:
B_plasma_markers = {
    ct: [m for m in ct_markers if m in adata.var.index]
    for ct, ct_markers in marker_genes.items()
    if ct in B_plasma_cts
}

In [ ]:
sc.pl.dotplot(
    adata,
    groupby="leiden_1",
    var_names=B_plasma_markers,
    standard_scale="var",
)

In [ ]:
cl_annotation = {
    "2": "Naive CD20+ B",
    "8": "Transitional B",
}

In [ ]:
adata.obs["manual_celltype_annotation"] = adata.obs.leiden_1.map(cl_annotation)

In [ ]:
sc.pl.umap(
    adata,
    color=["manual_celltype_annotation"]
)

### 3.4 Differential-expression-assisted annotation

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="leiden_1",
    method="wilcoxon",
    key_added="dea_leiden_1",
)

In [ ]:
sc.tl.dendrogram(
    adata,
    groupby="leiden_1",
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby="leiden_1",
    standard_scale="var",
    n_genes=5,
    key="dea_leiden_1",
)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata,
    min_in_group_fraction=0.2,
    max_out_group_fraction=0.2,
    key="dea_leiden_1",
    key_added="dea_leiden_1_filtered",
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby="leiden_1",
    standard_scale="var",
    n_genes=5,
    key="dea_leiden_1_filtered",
)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "CD247",
        "MYOM2",
        "KLRD1",
        "PRF1",
        "KLRF1",
        "leiden_1",
    ],
    vmax="p99",
    legend_loc="on data",
    frameon=False,
    cmap="Reds",
)

In [ ]:
cl_annotation["9"] = "NK cells (?)"

In [ ]:
adata.obs["manual_celltype_annotation"] = adata.obs.leiden_1.map(cl_annotation)

## 4. Automated annotation with CellTypist

Prepare normalized expression and apply coarse- and fine-grained immune reference models.

In [ ]:
adata_celltypist = adata.copy()

adata_celltypist.X = adata.layers["counts"]

sc.pp.normalize_total(
    adata_celltypist,
    target_sum=10**4
)

sc.pp.log1p(adata_celltypist)

adata_celltypist.X = adata_celltypist.X.toarray()

In [ ]:
models.download_models(
    force_update=True,
    model=[
        "Immune_All_Low.pkl",
        "Immune_All_High.pkl",
    ],
)

In [ ]:
model_low = models.Model.load(
    model="Immune_All_Low.pkl"
)

model_high = models.Model.load(
    model="Immune_All_High.pkl"
)

In [ ]:
model_high.cell_types

In [ ]:
model_low.cell_types

### 4.1 Coarse CellTypist model

In [ ]:
predictions_high = celltypist.annotate(
    adata_celltypist,
    model=model_high,
    majority_voting=True
)

In [ ]:
predictions_high_adata = predictions_high.to_adata()

In [ ]:
adata.obs["celltypist_cell_label_coarse"] = predictions_high_adata.obs.loc[
    adata.obs.index,
    "majority_voting"
]

adata.obs["celltypist_conf_score_coarse"] = predictions_high_adata.obs.loc[
    adata.obs.index,
    "conf_score"
]

### 4.2 Fine CellTypist model

In [ ]:
predictions_low = celltypist.annotate(
    adata_celltypist,
    model=model_low,
    majority_voting=True
)

In [ ]:
predictions_low_adata = predictions_low.to_adata()

In [ ]:
adata.obs["celltypist_cell_label_fine"] = predictions_low_adata.obs.loc[
    adata.obs.index,
    "majority_voting"
]

adata.obs["celltypist_conf_score_fine"] = predictions_low_adata.obs.loc[
    adata.obs.index,
    "conf_score"
]

### 4.3 Visualize and compare CellTypist predictions

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "celltypist_cell_label_coarse",
        "celltypist_conf_score_coarse",
    ],
    frameon=False,
    sort_order=False,
    wspace=1,
)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "celltypist_cell_label_fine",
        "celltypist_conf_score_fine",
    ],
    frameon=False,
    sort_order=False,
    wspace=1,
)

In [ ]:
sc.tl.dendrogram(
    adata,
    groupby="celltypist_cell_label_fine",
)

sc.pl.dendrogram(
    adata,
    groupby="celltypist_cell_label_fine"
)

### 4.4 Compare selected Leiden clusters with CellTypist labels

In [ ]:
pd.crosstab(
    adata.obs.leiden_1,
    adata.obs.celltypist_cell_label_fine
).loc[
    "2", :
].sort_values(ascending=False)

In [ ]:
pd.crosstab(
    adata.obs.leiden_1,
    adata.obs.celltypist_cell_label_fine
).loc[
    "9", :
].sort_values(ascending=False)

In [ ]:
pd.crosstab(
    adata.obs.leiden_1,
    adata.obs.celltypist_cell_label_fine
).loc[
    "8", :
].sort_values(ascending=False)

## 5. Reference mapping with scArches

Prepare the query gene set, map it into the pretrained scVI latent space, and inspect marker expression after mapping.

In [ ]:
adata_to_map = adata.copy()

for layer in list(adata_to_map.layers.keys()):
    if layer != "counts":
        del adata_to_map.layers[layer]

adata_to_map.X = adata_to_map.layers["counts"]

In [ ]:
adata_to_map.var["gene_names"] = adata_to_map.var.index
adata_to_map.var.set_index("gene_ids", inplace=True)

adata_to_map.var.head()

### 5.1 Load reference features

In [ ]:
reference_model_features = pd.read_csv(
    REFERENCE_DIR / "annotation_reference_features.csv",
    index_col=0,
)


In [ ]:
reference_model_features["gene_names"] = reference_model_features.index
reference_model_features.set_index("gene_ids", inplace=True)

In [ ]:
print(
    "Total number of genes needed for mapping:",
    reference_model_features.shape[0],
)

In [ ]:
print(
    "Number of genes found in query dataset:",
    adata_to_map.var.index.isin(reference_model_features.index).sum(),
)

### 5.2 Add missing reference genes and align feature order

In [ ]:
missing_genes = [
    gene_id
    for gene_id in reference_model_features.index
    if gene_id not in adata_to_map.var.index
]

In [ ]:
missing_gene_adata = sc.AnnData(
    X=csr_matrix(
        np.zeros(
            shape=(adata.n_obs, len(missing_genes))
        ),
        dtype="float32",
    ),
    obs=adata.obs.iloc[:, :1],
    var=reference_model_features.loc[missing_genes, :],
)

missing_gene_adata.layers["counts"] = missing_gene_adata.X

In [ ]:
if "PCs" in adata_to_map.varm.keys():
    del adata_to_map.varm["PCs"]

In [ ]:
adata_to_map_augmented = sc.concat(
    [adata_to_map, missing_gene_adata],
    axis=1,
    join="outer",
    index_unique=None,
    merge="unique",
)

In [ ]:
adata_to_map_augmented = adata_to_map_augmented[
    :, reference_model_features.index
].copy()

In [ ]:
bool(
    (
        adata_to_map_augmented.var.index
        == reference_model_features.index
    ).all()
)

In [ ]:
adata_to_map_augmented.var["gene_ids"] = (
    adata_to_map_augmented.var.index
)

adata_to_map_augmented.var.set_index(
    "gene_names",
    inplace=True,
)

### 5.3 Define the query batch and load the pretrained reference model

In [ ]:
adata_to_map_augmented.obs["batch"] = "s4d8"
adata_to_map_augmented.obs["batch"] = (
    adata_to_map_augmented.obs["batch"].astype("category")
)

In [ ]:
adata_to_map_augmented.obs.batch.unique()

In [ ]:
sys.modules["pandas.core.indexes.numeric"] = pandas_indexes_base
pandas_indexes_base.Int64Index = pd.Index
pandas_indexes_base.Float64Index = pd.Index

In [ ]:
annotation_ref_model_path = (
    REFERENCE_DIR / "annotation_reference_model.pt"
)

model_dir = Path("./reference_model")
model_dir.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy(
    annotation_ref_model_path,
    model_dir / "model.pt",
)


### 5.4 Map the query and train the adapted scArches model

In [ ]:
scarches_model = sca.models.SCVI.load_query_data(
    adata=adata_to_map_augmented,
    reference_model=str(model_dir),
    freeze_dropout=True,
)

In [ ]:
scarches_model.train(
    max_epochs=500,
    plan_kwargs={
        "weight_decay": 0.0
    },
)

In [ ]:
adata.obsm["X_scVI"] = (
    scarches_model.get_latent_representation()
)

### 5.5 Build a UMAP from the scVI latent representation

In [ ]:
sc.pp.neighbors(
    adata,
    use_rep="X_scVI",
)

sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "IGHD",
        "IGHM",
        "PRDM1",
    ],
    vmin=0,
    vmax="p99",
    sort_order=False,
    frameon=False,
    cmap="Reds",
)

## 6. Joint reference-query embedding

Load the reference latent embedding and combine it with the mapped query representation.

In [ ]:
ref_emb = sc.read_h5ad(
    REFERENCE_DIR / "annotation_reference_embedding.h5ad"
)


In [ ]:
ref_emb.obs["reference_or_query"] = "reference"

In [ ]:
ref_emb

In [ ]:
adata_emb = sc.AnnData(
    X=adata.obsm["X_scVI"],
    obs=adata.obs
)

In [ ]:
adata_emb.obs["reference_or_query"] = "query"

In [ ]:
adata_emb.obs["cell_type"] = None

### 6.1 Concatenate reference and query embeddings

In [ ]:
emb_ref_query = sc.concat(
    [ref_emb, adata_emb],
    axis=0,
    join="outer",
    index_unique=None,
    merge="unique",
)

In [ ]:
sc.pp.neighbors(emb_ref_query)
sc.tl.umap(emb_ref_query)

In [ ]:
sc.pl.umap(
    emb_ref_query,
    color=["reference_or_query"],
    sort_order=False,
    frameon=False,
)

### 6.2 Inspect reference cell-type structure

In [ ]:
sc.set_figure_params(figsize=(8, 8))

In [ ]:
sc.pl.umap(
    emb_ref_query,
    color=["cell_type"],
    sort_order=False,
    frameon=False,
    legend_loc="on data",
    legend_fontsize=10,
    na_color="black",
)

## 7. Weighted KNN label transfer

Transfer reference cell-type labels to query cells and quantify prediction uncertainty.

In [ ]:
knn_transformer = sca.utils.knn.weighted_knn_trainer(
    train_adata=ref_emb,
    train_adata_emb="X",
    n_neighbors=15,
)

In [ ]:
labels, uncert = sca.utils.knn.weighted_knn_transfer(
    query_adata=adata_emb,
    query_adata_emb="X",
    label_keys="cell_type",
    knn_model=knn_transformer,
    ref_adata_obs=ref_emb.obs,
)

In [ ]:
adata_emb.obs["transf_cell_type"] = labels.loc[
    adata_emb.obs.index, "cell_type"
]

adata_emb.obs["transf_cell_type_unc"] = uncert.loc[
    adata_emb.obs.index, "cell_type"
]

In [ ]:
adata.obs.loc[
    adata_emb.obs.index, "transf_cell_type"
] = adata_emb.obs["transf_cell_type"]

adata.obs.loc[
    adata_emb.obs.index, "transf_cell_type_unc"
] = adata_emb.obs["transf_cell_type_unc"]

adata.obs["transf_cell_type_unc"] = adata.obs[
    "transf_cell_type_unc"
].astype(float)

### 7.1 Visualize transferred labels and uncertainty

In [ ]:
sc.set_figure_params(figsize=(5, 5))

In [ ]:
sc.pl.umap(
    adata,
    color="transf_cell_type",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata,
    color="transf_cell_type_unc",
    frameon=False,
)

### 7.2 Summarize uncertainty by transferred cell type

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

ct_order = (
    adata.obs.groupby("transf_cell_type")
    .agg({"transf_cell_type_unc": "median"})
    .sort_values(
        by="transf_cell_type_unc",
        ascending=False,
    )
)

sns.boxplot(
    adata.obs,
    x="transf_cell_type",
    y="transf_cell_type_unc",
    color="grey",
    ax=ax,
    order=ct_order.index,
)

ax.tick_params(
    rotation=90,
    axis="x",
)

### 7.3 Mark high-uncertainty predictions as `Unknown`

In [ ]:
adata.obs["transf_cell_type_certain"] = (
    adata.obs.transf_cell_type.tolist()
)

adata.obs.loc[
    adata.obs.transf_cell_type_unc > 0.2,
    "transf_cell_type_certain",
] = "Unknown"

In [ ]:
sc.pl.umap(
    adata,
    color="transf_cell_type_certain",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata,
    color="transf_cell_type_certain",
    groups="Unknown",
)

## 8. Marker-based validation of transferred labels

In [ ]:
cell_types_to_check = [
    "CD14+ Mono",
    "cDC2",
    "NK",
    "B1 B",
    "CD4+ T activated",
    "T naive",
    "MK/E prog",
]

In [ ]:
sc.pl.dotplot(
    adata,
    var_names={
        ct: marker_genes_in_data[ct]
        for ct in cell_types_to_check
    },
    groupby="transf_cell_type_certain",
    standard_scale="var",
)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "transf_cell_type_unc",
        "transf_cell_type_certain",
    ],
    frameon=False,
)

## 9. Save the annotated dataset

In [ ]:
if "dea_leiden_1_filtered" in adata.uns:
    fields = adata.uns[
        "dea_leiden_1_filtered"
    ]["names"].dtype.names

    for field in fields:
        adata.uns[
            "dea_leiden_1_filtered"
        ]["names"][field] = adata.uns[
            "dea_leiden_1_filtered"
        ]["names"][field].astype(str)

In [ ]:
adata.write_h5ad(
    RESULTS_DIR / "s4d8_annotated.h5ad"
)
